# CBCT Tooth & Canal Segmentation — Google Colab Pro Training

Notebook train K-fold cross-validation trên Colab Pro với các tính năng:
- Mount Google Drive để lưu code + dữ liệu + checkpoint persistent
- **Lưu checkpoint mỗi 12 giờ** (ngoài best model theo dice) để sống sót qua disconnect
- **Auto-resume**: khi session bị ngắt, chạy lại cell training sẽ tự load checkpoint gần nhất
- Background TensorBoard để theo dõi loss/dice

## Chuẩn bị trước khi chạy

Upload lên Google Drive theo cấu trúc:
```
MyDrive/cbct_segmentation/
├── code/              # toàn bộ .py của project (config.py, dataset.py, train.py, ...)
└── data/
    └── raw/           # 4 file CBCT gốc + label
        ├── SLZ000.nii.gz
        ├── SLZ000-label.nii.gz
        └── ...
```

Script sẽ tự tạo `data/teeth/` (sau khi split) và `checkpoints/` (sau khi train).

## 1. Kiểm tra GPU

Colab Pro thường phân bổ **T4 / L4 / A100**. Với patch 96³ và nnU-Net:
- **T4 (16GB)**: batch_size=4 OK, ~40s/epoch
- **L4 (24GB)**: batch_size=6-8 OK, ~25s/epoch
- **A100 (40GB)**: batch_size=8+ OK, ~12s/epoch

Ước lượng thời gian 4-fold × 200 epoch trên T4: **~9-12 giờ**.

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# === Cấu hình đường dẫn ===
PROJECT_DIR = Path('/content/drive/MyDrive/cbct_segmentation')
CODE_DIR    = PROJECT_DIR / 'code'
RAW_DIR     = PROJECT_DIR / 'data' / 'raw'       # 4 file CBCT gốc
TEETH_DIR   = PROJECT_DIR / 'data' / 'teeth'     # sau khi split
CKPT_DIR    = PROJECT_DIR / 'checkpoints'        # lưu trên Drive (persistent)
LOG_DIR     = PROJECT_DIR / 'logs'

for p in [CODE_DIR, RAW_DIR, TEETH_DIR, CKPT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Copy code từ Drive về /content để import nhanh hơn
LOCAL_CODE = Path('/content/segment')
LOCAL_CODE.mkdir(exist_ok=True)
!cp -r {CODE_DIR}/*.py {LOCAL_CODE}/ 2>/dev/null || echo 'Chưa có code trên Drive - upload trước!'
!ls {LOCAL_CODE}

os.chdir(LOCAL_CODE)
import sys
sys.path.insert(0, str(LOCAL_CODE))

## 3. Cài đặt dependencies

In [ ]:
!pip install -q monai==1.3.0 nibabel pynrrd tensorboard
!pip install -q 'einops' 'timm'  # cần cho Swin-UNETR

## 4. Split CBCT thành răng riêng lẻ (chỉ chạy 1 lần)

Chạy cell dưới nếu `data/teeth/` chưa có data. Kết quả sẽ được lưu trên Drive nên không cần chạy lại khi restart.

In [ ]:
teeth_images = TEETH_DIR / 'images'
if teeth_images.exists() and any(teeth_images.iterdir()):
    print(f'Đã có {len(list(teeth_images.iterdir()))} răng trong {teeth_images}, skip split.')
else:
    !python split_teeth.py --batch --input_dir {RAW_DIR} --output {TEETH_DIR}
    print('\n=== Danh sách răng đã tách ===')
    !ls {teeth_images}

## 5. TensorBoard (chạy background)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## 6. Training K-fold với checkpoint mỗi 12h + auto-resume

Cell này bọc `KFoldTrainer` để:
- **Save `latest.pth` mỗi 12h** (dựa trên thời gian thực, không theo epoch)
- **Auto-resume**: nếu `latest.pth` tồn tại → load state (model + optimizer + epoch)
- **Vẫn giữ** cơ chế lưu `best_model.pth` theo val dice

Nếu Colab ngắt kết nối, chỉ cần chạy lại cell này — quá trình train sẽ tiếp tục từ checkpoint cuối cùng.

In [ ]:
import time
import json
import torch
from pathlib import Path

from config import AugmentConfig, DataConfig, ModelConfig, TrainConfig
from dataset import kfold_split_by_case, get_fold_dataloaders, prepare_data_list
from transforms import get_train_transforms, get_val_transforms
from train_kfold import KFoldTrainer

# === Siêu tham số (đã tune cho dataset 24 răng / 4 CBCT trên Colab T4) ===
ARCH            = 'nnunet'        # best cho medical segmentation với dataset nhỏ
N_FOLDS         = 4               # leave-one-case-out (4 CBCT → 4 fold)
EPOCHS          = 200             # đủ hội tụ với nnU-Net + 18 samples/fold
BATCH_SIZE      = 4               # T4 16GB OK với nnU-Net + patch 96³
LR              = 3e-4            # nnU-Net AdamW standard
WEIGHT_DECAY    = 3e-5
PATCH_SIZE      = (96, 96, 96)    # fit răng đơn sau split_teeth
CLASS_WEIGHTS   = [1.0, 1.0, 5.0] # bg, tooth, canal (canal x5)
EXPERIMENT      = 'kfold_nnunet_tooth_canal'
CKPT_INTERVAL_H = 12.0            # lưu latest.pth mỗi 12 giờ

# === Config chung ===
data_config  = DataConfig(data_dir=str(TEETH_DIR), patch_size=PATCH_SIZE)
model_config = ModelConfig(architecture=ARCH)
aug_config   = AugmentConfig()

train_tfm = get_train_transforms(data_config, aug_config)
val_tfm   = get_val_transforms(data_config)

data_list = prepare_data_list(str(TEETH_DIR))
print(f'\nTổng số răng: {len(data_list)}')

folds = kfold_split_by_case(data_list, n_folds=N_FOLDS, seed=data_config.seed)

kfold_root = CKPT_DIR / EXPERIMENT
kfold_root.mkdir(parents=True, exist_ok=True)
print(f'K-fold checkpoint root: {kfold_root}')
print(f'\nSiêu tham số:')
print(f'  Architecture:  {ARCH}')
print(f'  Epochs:        {EPOCHS}')
print(f'  Batch size:    {BATCH_SIZE}')
print(f'  Learning rate: {LR}')
print(f'  Patch size:    {PATCH_SIZE}')
print(f'  Class weights: {CLASS_WEIGHTS}')
print(f'  Save every:    {CKPT_INTERVAL_H}h')

In [ ]:
def train_fold_with_resume(fold_idx, train_data, val_data):
    """Train 1 fold với auto-resume + save latest mỗi 12h."""
    fold_num = fold_idx + 1
    fold_dir = kfold_root / f'fold{fold_num}'
    fold_dir.mkdir(parents=True, exist_ok=True)

    latest_path = fold_dir / 'latest.pth'
    done_path   = fold_dir / 'DONE'

    if done_path.exists():
        print(f'\n[Fold {fold_num}] Đã xong trước đó, skip. (xóa file DONE nếu muốn train lại)')
        ckpt = torch.load(fold_dir / 'best_model.pth', map_location='cpu')
        return ckpt.get('best_val_dice', 0.0)

    print(f'\n{"="*70}\nFOLD {fold_num}/{N_FOLDS}\n{"="*70}')
    print(f'  Train: {len(train_data)} răng ({sorted({d["case_id"] for d in train_data})})')
    print(f'  Val:   {len(val_data)} răng ({sorted({d["case_id"] for d in val_data})})')

    train_loader, val_loader = get_fold_dataloaders(
        train_data=train_data, val_data=val_data,
        train_transforms=train_tfm, val_transforms=val_tfm,
        batch_size=BATCH_SIZE, num_workers=2, oversample_canal=True,
        canal_oversample_ratio=3.0,
    )

    train_config = TrainConfig(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        num_workers=2,
        warmup_epochs=5,
        oversample_canal=True,
        canal_oversample_ratio=3.0,
        class_weights=CLASS_WEIGHTS,
        early_stopping_patience=25,
        checkpoint_dir=str(fold_dir),
        experiment_name=f'{EXPERIMENT}/fold{fold_num}',
        log_dir=str(LOG_DIR),
    )

    trainer = KFoldTrainer(
        data_config=data_config, model_config=model_config,
        train_config=train_config, aug_config=aug_config,
        train_loader=train_loader, val_loader=val_loader,
    )

    # === Auto-resume từ latest.pth ===
    start_epoch = 0
    if latest_path.exists():
        print(f'[Resume] Loading {latest_path}')
        ckpt = torch.load(latest_path, map_location=trainer.device)
        trainer.model.load_state_dict(ckpt['model_state_dict'])
        trainer.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            trainer.scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if 'scaler_state_dict' in ckpt:
            trainer.scaler.load_state_dict(ckpt['scaler_state_dict'])
        trainer.best_val_dice = ckpt.get('best_val_dice', 0.0)
        start_epoch = ckpt.get('epoch', 0) + 1
        print(f'[Resume] Tiếp tục từ epoch {start_epoch}, best_dice={trainer.best_val_dice:.4f}')

    # === Training loop với 12h checkpointing ===
    last_save_time = time.time()
    interval_sec = CKPT_INTERVAL_H * 3600
    epoch = start_epoch

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        train_metrics = trainer.train_epoch(epoch)
        val_metrics   = trainer.validate(epoch)
        elapsed       = time.time() - t0
        lr_now        = trainer.optimizer.param_groups[0]['lr']

        # TensorBoard logging
        trainer.writer.add_scalar('Loss/train', train_metrics['train_loss'], epoch)
        trainer.writer.add_scalar('Loss/val',   val_metrics['val_loss'],   epoch)
        trainer.writer.add_scalar('Dice/tooth', val_metrics['val_dice_tooth'], epoch)
        trainer.writer.add_scalar('Dice/canal', val_metrics['val_dice_canal'], epoch)
        trainer.writer.add_scalar('LR', lr_now, epoch)

        mean_dice = (val_metrics['val_dice_tooth'] + val_metrics['val_dice_canal']) / 2
        is_best = mean_dice > trainer.best_val_dice
        if is_best:
            trainer.best_val_dice = mean_dice
            trainer.patience_counter = 0
        else:
            trainer.patience_counter += 1

        print(f"Ep [{epoch+1}/{EPOCHS}] ({elapsed:.0f}s) lr={lr_now:.2e} | "
              f"train={train_metrics['train_loss']:.4f} val={val_metrics['val_loss']:.4f} | "
              f"dice_tooth={val_metrics['val_dice_tooth']:.4f} "
              f"dice_canal={val_metrics['val_dice_canal']:.4f}"
              + ('  *BEST*' if is_best else ''))

        # --- Save best ---
        if is_best:
            torch.save({
                'epoch': epoch,
                'model_state_dict': trainer.model.state_dict(),
                'optimizer_state_dict': trainer.optimizer.state_dict(),
                'scheduler_state_dict': trainer.scheduler.state_dict(),
                'best_val_dice': trainer.best_val_dice,
                'metrics': {**train_metrics, **val_metrics},
            }, fold_dir / 'best_model.pth')

        # --- Save latest mỗi 12h ---
        if time.time() - last_save_time >= interval_sec:
            torch.save({
                'epoch': epoch,
                'model_state_dict': trainer.model.state_dict(),
                'optimizer_state_dict': trainer.optimizer.state_dict(),
                'scheduler_state_dict': trainer.scheduler.state_dict(),
                'scaler_state_dict': trainer.scaler.state_dict(),
                'best_val_dice': trainer.best_val_dice,
            }, latest_path)
            last_save_time = time.time()
            print(f'  [Checkpoint 12h] Đã lưu {latest_path.name} tại epoch {epoch+1}')

        # --- Early stopping ---
        if trainer.patience_counter >= 25:
            print(f'\nEarly stopping tại epoch {epoch+1} (no improvement 25 epochs)')
            break

    # Save final latest + đánh dấu DONE
    torch.save({
        'epoch': epoch,
        'model_state_dict': trainer.model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
        'scheduler_state_dict': trainer.scheduler.state_dict(),
        'scaler_state_dict': trainer.scaler.state_dict(),
        'best_val_dice': trainer.best_val_dice,
    }, latest_path)
    done_path.write_text(f'best_val_dice={trainer.best_val_dice:.4f}\n')

    best_dice = trainer.best_val_dice
    trainer.writer.close()
    del trainer, train_loader, val_loader
    torch.cuda.empty_cache()
    return best_dice


# === Chạy toàn bộ K-fold ===
fold_dices = []
for fold_idx, (tr_data, va_data) in enumerate(folds):
    train_fold_with_resume(fold_idx, tr_data, va_data)

    # Đọc lại best dice từ best_model.pth
    best_ckpt = kfold_root / f'fold{fold_idx+1}' / 'best_model.pth'
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location='cpu')
        fold_dices.append({
            'fold': fold_idx + 1,
            'best_val_dice': ckpt.get('best_val_dice', 0.0),
            'val_cases': sorted({d['case_id'] for d in va_data}),
        })

# === Tổng hợp summary ===
if fold_dices:
    import numpy as np
    dices = [f['best_val_dice'] for f in fold_dices]
    summary = {
        'experiment': EXPERIMENT,
        'architecture': ARCH,
        'n_folds': len(fold_dices),
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LR,
        'class_weights': CLASS_WEIGHTS,
        'mean_dice': float(np.mean(dices)),
        'std_dice':  float(np.std(dices)),
        'folds': fold_dices,
    }
    with open(kfold_root / 'kfold_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print(f'\n{"="*70}\nK-FOLD HOÀN TẤT\n{"="*70}')
    for r in fold_dices:
        print(f"  Fold {r['fold']}: dice={r['best_val_dice']:.4f} (val={r['val_cases']})")
    print(f"\n  Mean dice: {summary['mean_dice']:.4f} ± {summary['std_dice']:.4f}")
    print(f"  Summary: {kfold_root / 'kfold_summary.json'}")

## 7. Inference với fold tốt nhất

Sau khi training xong, chọn tự động fold có val dice cao nhất.

In [ ]:
TEST_INPUT_DIR = PROJECT_DIR / 'data' / 'test_images'
PRED_OUTPUT_DIR = PROJECT_DIR / 'predictions'
PRED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

!python inference.py \
    --kfold_dir {kfold_root} \
    --input_dir {TEST_INPUT_DIR} \
    --output_dir {PRED_OUTPUT_DIR} \
    --arch nnunet

## 8. (Tùy chọn) Ngăn Colab idle disconnect

Chạy cell JS sau trong **Console của browser** (F12 → Console) để auto-click mỗi 60s:
```javascript
function ClickConnect(){
  console.log('Keeping alive...');
  document.querySelector('colab-toolbar-button#connect')?.click();
}
setInterval(ClickConnect, 60000);
```

**Lưu ý Colab Pro:**
- Session tối đa ~24h, idle timeout ~90 phút
- Nếu bị ngắt, chỉ cần mở lại notebook và **chạy lại cell Training** — auto-resume từ `latest.pth`
- Checkpoint lưu trên Drive nên không bị mất khi runtime bị recycle
- Nếu muốn train lại 1 fold từ đầu: xóa file `DONE` và `latest.pth` trong `checkpoints/.../foldN/`